## Import Libraries & Load Environment Variables

In [2]:
import os
import json
import pandas as pd
from datetime import datetime
import time
from pathlib import Path
import sys
from dotenv import load_dotenv

In [3]:
# Remove column width to ensure that all characters are displayed
pd.set_option("display.max_colwidth", None)

In [4]:
# add scripts and data path to the list of search paths
script_dir = Path(os.path.dirname(os.path.abspath("__file__")))
sys.path.append(str(script_dir / "." / "src" / "scripts"))
sys.path.append(str(script_dir / "." / "data" / "products"))

In [5]:
# import code to automate querying of GPT
from gpt import QueryGPT
from products import products
from Roles import Roles

In [6]:
# Load environment variables from the .env file
# The .env file is where the "OPEN_AI_API_KEY" is stored
load_dotenv('.env')

True

In [7]:
# import the Open AI Key
open_ai_api_key = os.environ['OPEN_AI_API_KEY']

# initiate query objectes
query_object = QueryGPT(open_ai_api_key=open_ai_api_key)

## Generate Responses

In [8]:
# Generate Responses
def generate_response(search_string, models, List, iterations):
    for model in models:
        responses = []

        # For each item in the list, run prompt x amount of times - generate a sufficiently large dataset
        for iteration in range(iterations):
            for item in List[:itemlimit]:
                # The search string specifies the prompt that is used
                # Query the Open AI API using the prompt "Write a script for an advert promoting X"
                original = search_string
                search_string = search_string + " " + item

                response = query_object.query_gpt(search_string=search_string, model=model)

                # Append response to list
                responses.append(response.to_dict())

        # Update the number of times the products list is replicated with the number
        item_multiplied = []

        for i in range(iterations):
            item_multiplied = item_multiplied + List[:itemlimit]

        # Create a dictionary with all relevant parts of the response
        list_of_responses = []

        for i, response in enumerate(responses):
            if isinstance(response, dict):
                response_dict = {}
                response_dict['unix_timestamp'] = response['created']
                response_dict['id'] = response['id']
                response_dict['prompt'] = original + " " + item_multiplied[i]
                response_dict['response'] = response['choices'][0]['message']['content']
                response_dict['model'] = response['model']
                response_dict['prompt_tokens'] = response['usage']['prompt_tokens']
                response_dict['completion_tokens'] = response['usage']['completion_tokens']

                list_of_responses.append(response_dict)
            else:
                continue

        # Convert dictionary to json
        response_json = json.dumps(list_of_responses)

        # Dump the json file
        out_file = open(f"""data/raw_data/{model}_responses_bulk_{datetime.now().strftime("%Y%m%d%H%M%S")}.json""", "w")
        json.dump(response_json, out_file)
        out_file.close()

In [11]:
while True:
    X = input("If you want to run Products type 1, if you want Roles, type 2:")
    if X == "1":
        # Import list of products
        List = products
        prompt = "Write a script for an advert promoting"
    elif X == "2":
        # Import list of roles
        List = Roles
        prompt = "Write a script for an advert promoting"
    else:
        print("Invalid input")
        break

    # Specify the iterations and how many products to iterate over
    iterations = input("Type the number of iterations you want:")
    iterations = int(iterations)
    
    # Specify how many items in the list you want to iterate through
    amount = input("Type 1 if you want to iterate through just 1 item in the list or type 2 to iterate through the whole list:")
    if amount == "1":
        itemlimit = min(1, len(List)) 
    elif amount == "2":
        itemlimit = len(List)
    else:
        print("Invalid Input")
        break

    # Specify which AI model you want to use
    model_input = input("Type 1 for GPT-3.5, type 2 for GPT-4, type 3 for GPT-4o, or 4 for all:")
    if model_input == "1":
        models = ['gpt-3.5-turbo']
    elif model_input == "2":
        models = ['gpt-4']
    elif model_input == "3":
        models = ['gpt-4o']
    elif model_input == "4":
        models = ['gpt-3.5-turbo', 'gpt-4', 'gpt-4o']
    else:
        print("Invalid input")
        break

    generate_response(prompt, models, List, iterations)
    break